In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

## Feature Engineering & Preprocessing (One-Hot + Separate Scaling)

In [3]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# Add the scripts folder to path
sys.path.append(os.path.abspath("../scripts"))

# Import our feature engineering functions
from feature_engineering import *
from eda_functions import *
from gan_util import build_feature_info

## 1. Load raw data

In [4]:
df = load_data("../data/raw/german.csv", clean_cols=True, drop_unnamed=True)

# Preview
print("Raw data shape:", df.shape)
df.head()

Raw data shape: (1000, 21)


,CheckingAccountStatus,Duration,CreditHistory,Purpose,CreditAmount,SavingsAccount,EmploymentSince,InstallmentRate,PersonalStatusSex,OtherDebtors,...,Property,Age,OtherInstallmentPlans,Housing,ExistingCredits,Job,Dependents,Telephone,ForeignWorker,Class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


## 2. Target mapping & missing values

In [5]:
df['Class'] = df['Class'].map({1: 0, 2: 1})

df['Class'].unique()
df['Class'].value_counts()


Class
0    700
1    300
Name: count, dtype: int64

In [6]:

df = handle_missing_values(df)

No missing values detected. Skipping imputation.


## 3. Feature engineering
### Ratio: CreditAmount / Duration

In [7]:
df = create_ratio(df, 'CreditAmount', 'Duration', 'Credit_per_Duration')
df[['CreditAmount', 'Duration', 'Credit_per_Duration']].head()


,CreditAmount,Duration,Credit_per_Duration
0,1169,6,194.833333
1,5951,48,123.979167
2,2096,12,174.666667
3,7882,42,187.666667
4,4870,24,202.916667


### Binning: Age into groups

In [8]:
df = bin_column(df, 'Age', bins=[18, 25, 40, 60, 100],
                labels=['Young', 'Adult', 'Mature', 'Senior'],
                new_col='AgeGroup')
df[['Age', 'AgeGroup']].head()

,Age,AgeGroup
0,67,Senior
1,22,Young
2,49,Mature
3,45,Mature
4,53,Mature


## 4. Identify numerical / categorical columns

In [10]:
num_cols, cat_cols = get_feature_types(df, target='Class')
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

Numerical: ['Duration', 'CreditAmount', 'InstallmentRate', 'ResidenceSince', 'Age', 'ExistingCredits', 'Dependents', 'Credit_per_Duration']
Categorical: ['CheckingAccountStatus', 'CreditHistory', 'Purpose', 'SavingsAccount', 'EmploymentSince', 'PersonalStatusSex', 'OtherDebtors', 'Property', 'OtherInstallmentPlans', 'Housing', 'Job', 'Telephone', 'ForeignWorker', 'AgeGroup']


## 5. Auto‑detect low‑cardinality numerical columns as categorical

In [11]:
## Automatically detect low‑cardinality numerical columns as categorical
num_cols, cat_cols = auto_detect_categorical(df, num_cols, cat_cols, unique_threshold=10)
print("\nAfter auto‑detection:")
print("Numerical (continuous):", num_cols)
print("Categorical (including detected):", cat_cols)

Moving 'InstallmentRate' to categorical (unique values: 4 ≤ 10)
Moving 'ResidenceSince' to categorical (unique values: 4 ≤ 10)
Moving 'ExistingCredits' to categorical (unique values: 4 ≤ 10)
Moving 'Dependents' to categorical (unique values: 2 ≤ 10)

After auto‑detection:
Numerical (continuous): ['Duration', 'CreditAmount', 'Age', 'Credit_per_Duration']
Categorical (including detected): ['CheckingAccountStatus', 'CreditHistory', 'Purpose', 'SavingsAccount', 'EmploymentSince', 'PersonalStatusSex', 'OtherDebtors', 'Property', 'OtherInstallmentPlans', 'Housing', 'Job', 'Telephone', 'ForeignWorker', 'AgeGroup', 'InstallmentRate', 'ResidenceSince', 'ExistingCredits', 'Dependents']


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   CheckingAccountStatus  1000 non-null   object  
 1   Duration               1000 non-null   int64   
 2   CreditHistory          1000 non-null   object  
 3   Purpose                1000 non-null   object  
 4   CreditAmount           1000 non-null   int64   
 5   SavingsAccount         1000 non-null   object  
 6   EmploymentSince        1000 non-null   object  
 7   InstallmentRate        1000 non-null   category
 8   PersonalStatusSex      1000 non-null   object  
 9   OtherDebtors           1000 non-null   object  
 10  ResidenceSince         1000 non-null   category
 11  Property               1000 non-null   object  
 12  Age                    1000 non-null   int64   
 13  OtherInstallmentPlans  1000 non-null   object  
 14  Housing                1000 non-null   ob

## 5. One-hot encode categorical variables

In [13]:
df_encoded, cat_cols = encode_categorical(df)   # works because columns are already 'category'
print("Shape after one‑hot:", df_encoded.shape)


# Verify that InstallmentRate was one‑hot expanded
print([c for c in df_encoded.columns if 'InstallmentRate' in c])
df_encoded.head()


Shape after one‑hot: (1000, 77)
['InstallmentRate_1', 'InstallmentRate_2', 'InstallmentRate_3', 'InstallmentRate_4']


,Duration,CreditAmount,Age,Class,Credit_per_Duration,CheckingAccountStatus_A11,CheckingAccountStatus_A12,CheckingAccountStatus_A13,CheckingAccountStatus_A14,CreditHistory_A30,...,Dependents_1,Dependents_2,Telephone_A191,Telephone_A192,ForeignWorker_A201,ForeignWorker_A202,AgeGroup_Young,AgeGroup_Adult,AgeGroup_Mature,AgeGroup_Senior
0,6,1169,67,0,194.833333,True,False,False,False,False,...,True,False,False,True,True,False,False,False,False,True
1,48,5951,22,1,123.979167,False,True,False,False,False,...,True,False,True,False,True,False,True,False,False,False
2,12,2096,49,0,174.666667,False,False,False,True,False,...,False,True,True,False,True,False,False,False,True,False
3,42,7882,45,0,187.666667,True,False,False,False,False,...,False,True,True,False,True,False,False,False,True,False
4,24,4870,53,1,202.916667,True,False,False,False,False,...,False,True,True,False,True,False,False,False,True,False


## 6. Build feature_info for GAN

In [14]:
feature_info = build_feature_info(df_encoded, cat_cols, target_col='Class')
column_order = df_encoded.drop(columns=['Class']).columns.tolist()

print("Numerical features:", feature_info["numerical"])
print("Categorical groups:", list(feature_info["categorical"].keys()))

Numerical features: ['Duration', 'CreditAmount', 'Age', 'Credit_per_Duration']
Categorical groups: ['CheckingAccountStatus', 'CreditHistory', 'Purpose', 'SavingsAccount', 'EmploymentSince', 'InstallmentRate', 'PersonalStatusSex', 'OtherDebtors', 'ResidenceSince', 'Property', 'OtherInstallmentPlans', 'Housing', 'ExistingCredits', 'Job', 'Dependents', 'Telephone', 'ForeignWorker', 'AgeGroup']


## 7. Save everything

In [16]:
df_encoded.to_csv("../data/processed/german_processed_onehot.csv", index=False)
joblib.dump(feature_info, "../models/feature_info.pkl")
joblib.dump(column_order, "../models/column_order.pkl")
joblib.dump(cat_cols, "../models/cat_cols.pkl")
print("All saved.")

All saved.
